# 08_test — 봉인한 test를 한 번 열어 평가 (EV-10 · RT-08)

[최종 실험 계획서](../../../docs/ml/ROUTER_FINAL_EXPERIMENT_PLAN.md) 2.5·3·4.4절. 07b lock(`lock/selection-lock.json`)에 고정한 설정·보정·ρ·비용 목표·λ를 그대로 쓰고, **여기서는 아무것도 고르지 않는다.**

- **갈래 ① 비용 고려:** 라우터마다 규칙(상향 고려·한 번에 고르기) × κ(0~1) × 비용 수준(25·50·75%)에서, test 효용(최종 해결 − κ × 실패 횟수) − 같은 test 비용의 고정 시작 단계(혼합 포함) 값 = 이득. 질문 묶음 단위 paired bootstrap 10,000회로 95% 구간. κ\* = 이득 > .01이고 구간 하한 > 0인 가장 작은 κ.
- 기준 셋: Claude 1:3:5(주), 집단 평균 데이터 가격(민감도), A test v1 채점(점 추정, H6).
- **갈래 ② τ 규칙:** 보정 확률 ≥ τ인 가장 싼 단계에서 시작. 필요 단계 정확도, 첫 시도 성공, 최종 해결, 호출 수, 비용(τ .5~.95).
- 가설 H1~H7과 RT-08 판정을 계획서의 사전 규칙대로 계산한다(`hypotheses.json`).
- 영어가 아닌 문항 246건은 07b와 같이 뺀다.

A test의 v1 채점값과 bootstrap 묶음 ID는 Drive `inputs/test-data/`(원본 파일 11개, 87MB, SHA 목록 포함)에서 읽는다.

**런타임: CPU면 충분하다.** 리허설 몇 분, test 20~30분. **test는 한 번만 연다.** 출력 폴더 `test/`가 이미 있으면 멈춘다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys, zipfile
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
SOURCE_ZIP = RUN / 'code' / 'router-final-source-test.zip'
expected = SOURCE_ZIP.with_suffix('.sha256').read_text().split()[0]
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == expected, '소스 ZIP이 바뀌었다'
REPO = Path('/content/router-final-source')
with zipfile.ZipFile(SOURCE_ZIP) as z:
    for name in z.namelist():
        assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
    z.extractall(REPO)
for name, digest in json.loads((REPO / 'source-bundle-manifest.json').read_text()).items():
    assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == digest, name
DATA_ROOT = RUN / 'inputs' / 'test-data'
for name, digest in json.loads((DATA_ROOT / 'test-data-manifest.json').read_text())['files'].items():
    assert hashlib.sha256((DATA_ROOT / name).read_bytes()).hexdigest() == digest, name
assert sys.version_info[:2] in {(3, 11), (3, 12)}, '런타임 버전을 2026.07(Python 3.12)로 바꾸세요'
subprocess.run([sys.executable, '-m', 'pip', 'install', 'scikit-learn==1.6.1', 'numpy==2.1.2', 'pandas==2.3.3', 'pyarrow==21.0.0'], check=True)

설치가 끝나면 **런타임 → 세션 다시 시작**을 한 번 한 뒤 아래 셀부터 실행합니다.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
REPO = Path('/content/router-final-source')
DATA_ROOT = RUN / 'inputs' / 'test-data'

def run_logged(args):
    """Run test_eval.py and stream its log into this cell (subprocess.run hides it in Colab)."""
    proc = subprocess.Popen([sys.executable, str(REPO / 'ml/notebooks/router_final/test_eval.py'), '--run-dir', str(RUN),
                             '--repo', str(REPO), '--data-root', str(DATA_ROOT), *map(str, args)],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env=dict(os.environ, PYTHONUNBUFFERED='1'))
    for line in proc.stdout:
        print(line, end='')
    if proc.wait():
        raise RuntimeError(f'실패(exit {proc.returncode}). 위 로그의 마지막 부분(Traceback)을 보내 주세요')

## 1. 리허설 (dev, test를 읽지 않음)

dev 결과로 같은 코드를 돌려 경로와 환경을 확인한다. bootstrap 1,000회, 몇 분. 결과는 Colab 디스크(`/content`)에만 쓰고, dev에서 맞춘 보정을 dev에 다시 쓴 값이라 낙관적이므로 해석하지 않는다.

In [ ]:
REHEARSAL = Path('/content/rehearse-08')
assert not REHEARSAL.exists(), '이미 리허설을 했다(세션을 새로 시작했다면 다시 해도 된다)'
run_logged(['--output-dir', REHEARSAL, '--rehearse-on-dev', '--iterations', 1000])
print((REHEARSAL / 'test-summary.md').read_text()[:1500])

## 2. test 열기 (한 번만)

07b lock 요약을 보고하고 확인을 받은 뒤 실행한다. bootstrap 10,000회.

In [ ]:
TEST = RUN / 'test'
assert not TEST.exists(), '이미 test를 열었다. 실패한 실행을 다시 할 때만 폴더를 다른 이름으로 옮긴 뒤 실행'
run_logged(['--output-dir', TEST])

In [ ]:
from pathlib import Path
import json
TEST = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001/test')
print((TEST / 'test-summary.md').read_text())
manifest = json.loads((TEST / 'manifest.json').read_text())
print(json.dumps({k: manifest[k] for k in ('counts', 'seconds', 'outputs')}, indent=1))

## 다음

`test/`의 `test-summary.md`·`hypotheses.json`을 보고한다. 판정 경계(±.01) 근처 결과만 09_seeds로 seed를 더한다. 표·그림·보고서는 10_report가 만든다.